# 156-Qubit Quantum Capabilities

This notebook demonstrates Zipminator's quantum random number generation using IBM's 156-qubit quantum hardware (Marrakesh/Fez backends via qBraid).

## QRNG vs Classical RNG
True quantum randomness is fundamentally different from pseudo-random number generators. Quantum measurements are inherently unpredictable, providing entropy that no classical algorithm can replicate.

In [ ]:
# Quantum Random Number Generation
from zipminator.crypto.quantum_random import QuantumRandom

qr = QuantumRandom()

# Generate quantum random bytes
random_bytes = qr.random_bytes(32)
print(f"32 quantum random bytes: {random_bytes.hex()}")
print(f"Entropy source: {qr.source_info()}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate samples
n_samples = 10000

# Classical PRNG
classical = np.random.random(n_samples)

# Quantum-enhanced random (falls back to OS entropy if no hardware available)
from zipminator.crypto.quantum_random import QuantumRandom
qr = QuantumRandom()
quantum = np.array([qr.random() for _ in range(min(n_samples, 1000))])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(classical, bins=50, alpha=0.7, color='#3b82f6', label='Classical PRNG')
axes[0].set_title('Classical PRNG Distribution')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

axes[1].hist(quantum, bins=50, alpha=0.7, color='#10b981', label='Quantum RNG')
axes[1].set_title('Quantum RNG Distribution')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.suptitle('Classical vs Quantum Random Distribution', y=1.02, fontweight='bold')
plt.show()

## Entropy Quality: NIST SP 800-22

True randomness passes statistical tests that pseudo-random sequences may fail. The NIST SP 800-22 test suite evaluates randomness quality.

In [ ]:
import numpy as np

def monobit_test(bits: bytes) -> dict:
    """NIST SP 800-22 Frequency (Monobit) Test"""
    bit_string = ''.join(format(b, '08b') for b in bits)
    n = len(bit_string)
    s = sum(1 if b == '1' else -1 for b in bit_string)
    s_obs = abs(s) / np.sqrt(n)
    from scipy.special import erfc
    p_value = erfc(s_obs / np.sqrt(2))
    return {"test": "Monobit", "s_obs": s_obs, "p_value": p_value, "pass": p_value >= 0.01}

def runs_test(bits: bytes) -> dict:
    """NIST SP 800-22 Runs Test"""
    bit_string = ''.join(format(b, '08b') for b in bits)
    n = len(bit_string)
    ones = sum(1 for b in bit_string if b == '1')
    pi = ones / n
    if abs(pi - 0.5) >= 2 / np.sqrt(n):
        return {"test": "Runs", "p_value": 0.0, "pass": False}
    runs = 1 + sum(1 for i in range(1, n) if bit_string[i] != bit_string[i-1])
    num = abs(runs - 2 * n * pi * (1 - pi))
    den = 2 * np.sqrt(2 * n) * pi * (1 - pi)
    from scipy.special import erfc
    p_value = erfc(num / den)
    return {"test": "Runs", "p_value": p_value, "pass": p_value >= 0.01}

# Test with quantum random bytes
from zipminator.crypto.quantum_random import QuantumRandom
qr = QuantumRandom()
sample = qr.random_bytes(1024)

results = [monobit_test(sample), runs_test(sample)]
for r in results:
    status = "PASS" if r["pass"] else "FAIL"
    print(f"  {r['test']}: p-value={r['p_value']:.6f} [{status}]")

## Quantum Hardware Status

Zipminator harvests entropy from IBM Quantum backends:
- **IBM Marrakesh**: 156 qubits, Heron r2 processor
- **IBM Fez**: 156 qubits, Heron r2 processor
- **Fallback**: OS entropy via `secrets.token_bytes()` when hardware unavailable

The entropy pool at `quantum_entropy/quantum_entropy_pool.bin` grows automatically via the harvester script.